This notebook is made to convert stellar evolutionary tracks from Donnan, Rood, & O'Connell (1993) [hereafter DRO3] into isochrones, to match the format used by FSPS for the other isochrone libraries.  The original DRO3 evolutionary tracks were obtained through CDS (https://cdsarc.cds.unistra.fr/viz-bin/cat/J/ApJ/419/596)

In [66]:
# Import the necessary libraries
import numpy as np
import numpy.lib.recfunctions as rfn
from scipy import interpolate
from matplotlib import pyplot as plt

In [76]:
# Read in the DRO3 tracks
dtype_in = [
    ('Mtot', float),
    ('FeH', float),
    ('OFe', float),
    ('Y', float),
    ('age_zahb', float),
    ('Yc', float),
    ('logT', float),
    ('logL', float),
    ('logg', float),
    ('logR', float),
    ('Msh', float),
    ('logTc', float),
    ('logrhoc', float)
]

dro3 = np.loadtxt('DRO3/table', dtype=dtype_in)

feh_comps = [-2.26, -1.48, -0.47, 0.00, 0.39, 0.43, 0.58, 0.71]
comps = [np.where(dro3['FeH'] == compi)[0] for compi in feh_comps]

# Add metallicity from the [Fe/H] column (see table 2 in Brown et al. 1997)
droZ = np.zeros(len(dro3['Mtot']), dtype=[('Z', float)])
droZ[comps[0]] = 0.0001
droZ[comps[1]] = 0.0006
droZ[comps[2]] = 0.0060
droZ[comps[3]] = 0.0169
droZ[comps[4]] = 0.0400
droZ[comps[5]] = 0.0400
droZ[comps[6]] = 0.0600
droZ[comps[7]] = 0.0600
# also add core masses
droMc = np.zeros(len(dro3['Mtot']), dtype=[('Mc', float)])
droMc[comps[0]] = 0.495
droMc[comps[1]] = 0.485
droMc[comps[2]] = 0.475
droMc[comps[3]] = 0.469
droMc[comps[4]] = 0.464
droMc[comps[5]] = 0.454
droMc[comps[6]] = 0.458
droMc[comps[7]] = 0.434
droMe = np.zeros(len(dro3['Mtot']), dtype=[('Menv', float)])
droMe[:] = dro3['Mtot'] - droMc['Mc']
dro3 = rfn.merge_arrays((dro3, droMc, droMe, droZ), flatten=True)

dro3

array([(0.488, -1.48, 0.6, 0.247,  1.46289, 0.95, 4.447 , 1.1847,  5.6825, 10.0645, 0.4855, 8.0675, 4.3201, 0.485, 0.003, 0.0006),
       (0.488, -1.48, 0.6, 0.247,  5.54043, 0.9 , 4.4465, 1.1949,  5.6705, 10.0705, 0.4859, 8.0697, 4.3197, 0.485, 0.003, 0.0006),
       (0.488, -1.48, 0.6, 0.247, 10.23408, 0.85, 4.4459, 1.2061,  5.657 , 10.0771, 0.486 , 8.072 , 4.3179, 0.485, 0.003, 0.0006),
       ...,
       (0.8  ,  0.71, 0. , 0.459, 98.81957, 0.  , 3.4614, 3.1281,  0.0113, 13.0074, 0.6299, 8.2435, 5.7997, 0.434, 0.366, 0.06  ),
       (0.8  ,  0.71, 0. , 0.459, 99.00128, 0.  , 3.4452, 3.2257, -0.151 , 13.0885, 0.6293, 8.243 , 5.8838, 0.434, 0.366, 0.06  ),
       (0.8  ,  0.71, 0. , 0.459, 99.02399, 0.  , 3.4434, 3.2386, -0.1713, 13.0986, 0.6293, 8.2426, 5.8955, 0.434, 0.366, 0.06  )],
      dtype=[('Mtot', '<f8'), ('FeH', '<f8'), ('OFe', '<f8'), ('Y', '<f8'), ('age_zahb', '<f8'), ('Yc', '<f8'), ('logT', '<f8'), ('logL', '<f8'), ('logg', '<f8'), ('logR', '<f8'), ('Msh', '<f8'), ('log

In [77]:
# Each new evolutionary track is delineated by a point where the time column will jump backwards, starting a new track
tracks = np.where(np.diff(dro3['age_zahb']) < 0.)[0]
ntrack = len(tracks) + 1

# We should have 136 total tracks
print(f'{ntrack=}')
tracks

ntrack=136


array([   68,   139,   212,   285,   351,   474,   750,   905,  1043,
        1108,  1236,  1289,  1405,  1448,  1539,  1586,  1660,  1876,
        1954,  2092,  2226,  2478,  2746,  3053,  3118,  3221,  3281,
        3337,  3437,  3493,  3584,  3644,  3684,  3800,  3911,  3957,
        4006,  4073,  4244,  4413,  4583,  4636,  4685,  4746,  4801,
        4861,  4918,  4963,  5054,  5144,  5265,  5318,  5367,  5421,
        5531,  5597,  5659,  5812,  5933,  5994,  6034,  6122,  6163,
        6203,  6246,  6293,  6333,  6394,  6454,  6516,  6580,  6649,
        6722,  6797,  6870,  6947,  7022,  7098,  7171,  7235,  7289,
        7331,  7374,  7417,  7479,  7544,  7610,  7677,  7747,  7820,
        7897,  7973,  8048,  8174,  8224,  8290,  8356,  8419,  8490,
        8556,  8614,  8674,  8733,  8799,  8869,  8939,  9010,  9083,
        9155,  9224,  9293,  9366,  9440,  9516,  9562,  9605,  9648,
        9690,  9743,  9811,  9875,  9940, 10006, 10073, 10140, 10206,
       10270, 10333,

In [78]:
# For each track, we need to create a linear interpolation along the time axis to sort them into isochrones as opposed to evolutionary tracks
# For each age value, we should have a variety of masses, temperatures, loggs, and compositions

# Output ages to be interpolated onto
age_lim = np.nanmin(dro3['age_zahb']), np.nanmax(dro3['age_zahb'])
age_out = np.arange(1., int(age_lim[1])+1., 1.)

# Prepare the output array for the isochrones
dtype_out = [
    ('age', float),
    ('Mtot', float),
    ('Mc', float),
    ('Menv', float),
    ('logL', float),
    ('logT', float),
    ('logTc', float),
    ('logg', float),
    ('logR', float),
    ('logrhoc', float),
    ('Y', float),
    ('Yc', float),
    ('Z', float),
    ('FeH', float),
    ('OFe', float),
]
dtype_names = [tp[0] for tp in dtype_out]

# A different isochrone array for each unique age value and composition
isoc_out = [
[
    {key: [] for key in dtype_names}
    for _ in range(len(age_out))
] 
for _ in range(len(comps))
]

# Loop over evolutionary tracks
for ci in range(len(comps)):
    prevtrack = 0
    tracks = np.where(np.diff(dro3['age_zahb'][comps[ci]]) < 0.)[0]

    for ti, track in enumerate(tracks):

        # Create linear interpolations for each variable 
        for name in dtype_names[1:]:
            pout = np.interp(age_out, dro3['age_zahb'][comps[ci]][prevtrack:track], dro3[name][comps[ci]][prevtrack:track], 
                left=np.nan, right=np.nan)
            for ai in range(len(age_out)):
                if np.isfinite(pout[ai]):
                    isoc_out[ci][ai][name].append(pout[ai])
                    if name == 'Mc':
                        isoc_out[ci][ai]['age'].append(age_out[ai])

        prevtrack = track + 1

isoc_out


[[{'age': [],
   'Mtot': [],
   'Mc': [],
   'Menv': [],
   'logL': [],
   'logT': [],
   'logTc': [],
   'logg': [],
   'logR': [],
   'logrhoc': [],
   'Y': [],
   'Yc': [],
   'Z': [],
   'FeH': [],
   'OFe': []},
  {'age': [2.0,
    2.0,
    2.0,
    2.0,
    2.0,
    2.0,
    2.0,
    2.0,
    2.0,
    2.0,
    2.0,
    2.0,
    2.0,
    2.0],
   'Mtot': [0.498,
    0.5,
    0.505,
    0.51,
    0.515,
    0.52,
    0.53,
    0.54,
    0.56,
    0.58,
    0.6,
    0.64,
    0.72,
    0.78],
   'Mc': [0.495,
    0.495,
    0.495,
    0.495,
    0.495,
    0.495,
    0.495,
    0.495,
    0.495,
    0.495,
    0.495,
    0.495,
    0.495,
    0.495],
   'Menv': [0.0030000000000000027,
    0.0050000000000000044,
    0.010000000000000009,
    0.015000000000000013,
    0.020000000000000018,
    0.025000000000000022,
    0.03500000000000003,
    0.04500000000000004,
    0.06500000000000006,
    0.08499999999999996,
    0.10499999999999998,
    0.14500000000000002,
    0.2249999999999999

In [79]:
# Write the output
for ci in range(len(comps)):
    with open(f'DRO3/isoc_comp{ci}.dat', 'w') as f:
        for isochrone in isoc_out[ci]:
            if len(isochrone['age']) == 0:
                continue
            f.write('#   ' + "".join([f'{dn:10s}' for dn in dtype_names]) + '\n')
            for i in range(len(isochrone['age'])):
                f.write("".join([f'{isochrone[param][i]:10.4f}' for param in list(isochrone.keys())]) + '\n')

In [ ]:
# composition legend
with open(f'DRO3/zylegend.dat', 'w') as f:
    f.write('-2.26   0.0001   0.245\n')
    f.write('-1.48   0.0006   0.247\n')
    f.write('-0.47   0.0060   0.257\n')
    f.write(' 0.00   0.0169   0.288\n')
    f.write(' 0.39   0.0400   0.292\n')
    f.write(' 0.43   0.0400   0.356\n')
    f.write(' 0.58   0.0600   0.289\n')
    f.write(' 0.71   0.0600   0.459\n')
